# Introduction

This is part 3 of my filters for dummies note. I'm the dummy writing these things down as I learn them. These notes are for future dummy me, who is prone to forget things, and for other dummies that want to learn about filters too. 

In the previous section, we derived the rough form of the algorithm under the assumption that our problem is 1D. But we did so using a couple of assumptions that might not always be true:

1. The measurement is a direct observation of the state being estimated.
2. The Kalman gain \(K\) is treated as a scalar weight.

In this section we are going to remove the assumptions to get a more general form of the algorithm.


****Note:** I felt like the intuition in this section was a lot harder to grasp and write down simply. I'm not sure I did a good job.

## When $z$ and $x$ Are Not The Same Thing

Our original estimation update equation, in which we measure the difference between our estimate $x_i$ and the new information $z_i$, was

$$
{\Huge
\begin{align}
x_{i+1}^+ = x_{i + 1}^- + K (z_{i + 1} - x_{i + 1}^-)
\end{align}
}
$$

But now we must account for the fact that the measurement $z_i$ might not be a direct measurement of the state we care about. For example, maybe our sensor can only measure position, but the state we are estimating includes both position and velocity. 

$$
{\Huge
z = p, \quad x = \begin{bmatrix} p \\ v \end{bmatrix}
}
$$

In another case $z$ might be something related to, but completely different than $x$. For example, let's say the quantity we care about $x$ is position in cartesian coordinates, but our sensor reports range and bearing (polar coordinates). Now we have

$$
{\Huge
z = \begin{bmatrix} r \\ \theta \end{bmatrix}, \quad x = \begin{bmatrix} x \\ y \end{bmatrix}
}
$$

Directly subtracting cartesian coordinates from polar coordinates does not provide any meaningful information. It's like subtracting a distance measured in inches from a distance measured in centimeters, without first converting both quantities to the same units. 

So, to resolve this, we introduce the matrix $H$, which is a way to transform the state we care about into the same form as the measurement:

<br>
$$
{\Huge
\begin{align}
\boxed{x_{i+1}^+ = x_{i + 1}^- + K \left(z_{i + 1} - Hx_{i + 1}^- \right)}
\end{align}
}
$$
<br>


****Note:** The existence of $H$ depends on the existence of a linear relationship between the thing being measured and the thing we care about. In the polar coordinates example, the relationship is non-linear and would instead require a function $h(x)$ to perform the transform, as a H won't exist. 

## The Covariance Update Formula In Higher Dimensions

In the 1D example, the uncertainty update equation

$$
{\Huge
P_{i + 1}^+ = (1 - K)P_{i + 1}^-
}
$$

Moving to higher dimensions, $K$ becomes a matrix and you can't subtract a matrix from a scalar $(1 - K)$. Uncertainty, $P$, in higher dimensions is represented with a covarince matrix, rather than the scalar variance. In order to update our equation so that it works in more than one dimension, we can borrow some intuition we developed earlier. We proved that when one random variable $y$ is a **linear transformation** of another random variable $x$ that

$$
{\Huge
Var(y) = a^2Var(x)
}
$$

And that in higher dimensions this looked like

$$
{\Huge
\Sigma_y = A\Sigma_xA^T
}
$$
<br>

We can leverage this same idea when updating our covariance $P$ since

1. Our update is just a weighted average of the old information and the new.
2. We know that averaging is a linear transformation.

So we can assume that the relationship between the old covariance and the new must take the form

$$
{\Huge
P^+ = A P^- A^T
}
$$

But how do we determine what A is? Well, we can look at what the transformation was:

<br>
$$
{\Huge
\begin{align}
x_{i+1}^+ &= x_{i + 1}^- + K (z_{i + 1} - Hx_{i + 1}^- ) \\
&= x_{i + 1}^- + K z_{i + 1} - KHx_{i + 1}^- \\
&= x_{i + 1}^-(I - KH) + K z_{i + 1}
\end{align}
}
$$
<br>

So, we see that the transformation on $x_{i + 1}^-$ was $(I - KH)$ and so we get $A = (I - KH)$ and

<br>
$$
{\Huge
\begin{align}
P^+ & \approx (I - KH) P^- (I - KH)^T
\end{align}
}
$$
<br>

But what about that $K z_{i + 1}$ term in the update formula above. How does that affect the output uncertainty? Well, $K$ is also a linear transformation, but it is applied to the measurement $z_{i + 1}$, which we know has uncertainty $R$. Therefore, we apply the same exact rule to it and get. 

$$
{\Huge
\boxed{P^+ = (I - KH) P^- (I - KH)^T + KRK^T}
}
$$
<br>

## Defining K in Higher Dimensions

Or original formulation of K, for one dimension was 

$${\Huge \frac{P}{P + R} }$$ 

But $P$ and $R$ are no longer single values but rather covariance matrices. These matrices tell us about our uncertainty in multiple variables, as well as how changing the uncertainty in one variable affects the uncertainty in another variable. For this reason, they are called covariance matrices. 

The old, simple intuition of $K = \frac{P}{P + R}$ being the fraction of total uncertainty associated with our estimate lacks some nuance, and the algebra itself needs to change

Let's start by addressing the denominator $\frac{1}{P + R}$. It may be the case that our sensor measures something different but related to the state of interest. For example, our state of interest might be position and velocity

$$
{\Huge
x = \begin{bmatrix} p \\ v \end{bmatrix}
}
$$

While our sensor might only measure position

$$
{\Huge
y = p 
}
$$

These two vectors have different shapes (2 x 1) versus (1 x 1), and likewise their covariance matrices may have different shapes. Just like you can't add $x + y$ together, you won't necessisarily be able to add $P + R$ together. We need to first transform one into the space of the other before we can meaningly compare them. This is much like converting two measurements into the same units before comparing them side by side. 

Recall that

1. $H$ was the matrix we defined that maps us from state estimation space to measurement space.
2. When talking about transforming covariance $\Sigma$ of a random variable by $A$, we use the form $A\Sigma A^T$.

So then let's perform that transformation:

<br>
$$
{\Huge
\frac{1}{P + R} \rightarrow \frac{1}{HPH^T + R}
}
$$
<br>

In the 1-D example, we viewed this fraction as an update weight derived from the ratio of estimate uncertainty to total uncertainty. That intuition still holds here. If either uncertainty increases, our trust goes down and therefore the weight of the update goes down. 

But notice that the ratio of $\frac{1}{HPH^T + R}$ is a measurement existing in the measurement space of $z$ and $R$, while our update is in the state estimation space of $x$ and $P$.

<br>
$$
{\Huge
x^+_{i + 1} = x^-_i + K(z_i - H x^-_i)
}
$$
<br>

How do we map this level of "trust" to our state estimation? Recall that a covariance matrix encodes informations about how certainty in one variable affects certainty in another variable, while $H$ is a matrix that encodes the relationship between the measurement and the state of interest. 

Using the example of state 

$$
{\Huge
x = \begin{bmatrix} p \\ v \end{bmatrix}
}
$$

and measurement

$$
{\Huge
y = p 
}
$$

We see that the matrix $H = \begin{bmatrix} 1 & 0 \end{bmatrix}$ will take us from the space of our state $x$ to the space of our measurement $y$

<br>
$$
{\Huge
\begin{bmatrix} 1 & 0 \end{bmatrix} \begin{bmatrix} p \\ v \end{bmatrix} = p
}
$$
<br>

But, what happens if we multiply our covariance $P$ by $H$? Let's say our variance in position is 20, our variance in velocity is 10 and our covariance between position and velocity is 5. That gives us

<br>
$$
{\Huge
 P = \begin{bmatrix} 20 & 5 \\ 5 & 10 \end{bmatrix}
}
$$
<br>

Then multiplying this by $H$ yields

<br>
$$
{\Huge
PH^T = \begin{bmatrix} 20 & 5 \\ 5 & 10 \end{bmatrix} \begin{bmatrix} 1 & 0 \end{bmatrix} = \begin{bmatrix} 20 \\ 5 \end{bmatrix}
}
$$
<br>

Where 20 is the variance of position, and 5 is the covariance between position and velocity. The non-zero covariance tells us how knowledge of errors in position can inform us about the errors in velocity. It is this knowledge that provides us the final intuition. We see that $PH^T$ in 

<br>
$$
{\Huge
\boxed{K = PH^T (HPH^T + R)^{-1}}
}
$$
<br>

gives us a way to map that weight into each of the elements of the state estimate. The matrix $PH^T$ uses the covariance structure encoded in P to determine how a residual in the measured quantity should influence each element of the state estimate.

# The Full Algorithm

1. Compute estimate at time $t_{i+1}$ with predictive model $$x_{i + 1}^- = f(x_i^+)$$
2. Compute estimate's covariance $$P_{i + 1}^- = JP_i^+J^T + Q$$
4. Use new information $z_{i + 1}$ with covariance $R$ to compute updated estimate $$x_{i + 1}^+ = x_{i + 1}^- + K(z_{i + 1} - Hx_{i + 1}^-)$$ where $$K = PH^T (HPH^T + R)^{-1}$$
5. Compute updated estimate's covariance $$P_{i+1}^+ = (I - KH)P(I - KH)^T + KRK^T$$

<br>
Where the notation above is defined by the following conventions:

\begin{align}
^- &:= \text{Defines a value at time } i \text{ before it is updated with new information}\\
^+ &:= \text{Defines a value at time } i \text{ after it is updated with new information}\\ 
J &:= \text{Jacobian of predictive model generating estimates: } x_{i + 1} = f(x_i) \\
Q &:= \text{The model output's covariance due to intrinsic errors in the model} \\
K &:= \text{Kalman Gain - coefficient matrix mapping residuals to update state } \\
H &:= \text{Measurement Matrix - transforms estimate into measurement space}
\end{align}
